# QM 640 Capstone — Step 5b: Firm-Level Financial Variables (SEC XBRL)

Addresses the Interim Report feedback: *"The study relies primarily on SEC EDGAR announcements and Yahoo Finance data. Additional market, financial, or firm-level variables could strengthen the explanatory power of the analysis."*

Pulls structured financial-statement data from SEC's own free XBRL Company Facts API (`data.sec.gov/api/xbrl/companyfacts`) — the same EDGAR system already used in Step 1, just a different endpoint, so no new data-source justification is needed. Produces:

- `log_assets` — alternative firm-size measure (complements `firm_size_log`/market cap)
- `leverage_ratio` — Liabilities / Assets
- `rd_intensity` — R&D expense / Revenue
- `prior_ai_disclosure_count` — self-generated, no API call: how many earlier confirmed AI events this same firm already had before this one

**Point-in-time correctness:** each event gets financials as reported *before that event's own file_date*, not one static value reused across a firm's multiple events - a firm's second AI announcement six months after its first should see updated financials, not the same snapshot.

**Run this after `05_market_model_car.ipynb`, before `06_statistical_tests.ipynb`.** It reads `analysis_dataset.csv` (built by Step 5, on the finalized `03g`/`03f`-corrected event set), merges the new financial variables directly into it, and re-saves it - so Step 6 picks them up automatically with no separate merge step.

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every other cell in this notebook reads from and writes to.

In [1]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 1115, done.
remote: Counting objects: 100% (302/302), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 1115 (delta 123), reused 240 (delta 90), pack-reused 813 (from 1)
Receiving objects: 100% (1115/1115), 8.20 MiB | 13.37 MiB/s, done.
Resolving deltas: 100% (564/564), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [2]:
!pip install -q pandas numpy requests

## Cell 3 — Configuration

In [3]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
OUTPUT_FILE = os.path.join(RAW_DIR, "firm_financials.csv")
ANALYSIS_DATASET_FILE = os.path.join(BASE_DIR, "data/processed/analysis_dataset.csv")

HEADERS = {"User-Agent": "QM640 Capstone research Shan_muganathan@yahoo.com"}  # edit to your real email

# Point-in-time lookups only use these SEC form types, never anything filed after the event
VALID_FORMS = ("10-K", "10-Q")

## Cell 4 — Load confirmed events

Uses the finalized `is_genuine_ai_event == "Y"` set, after `03g`'s reclassification review has been merged via `03f`. If `analysis_dataset.csv` doesn't exist yet, run `05_market_model_car.ipynb` first — this notebook merges into it, it doesn't create it.

In [4]:
import pandas as pd

screening = pd.read_csv(SCREENING_FILE)
screening["file_date"] = pd.to_datetime(screening["file_date"])

confirmed = screening[screening["is_genuine_ai_event"].astype(str).str.upper() == "Y"].copy()
confirmed = confirmed.dropna(subset=["cik", "ticker"])

print(f"Confirmed events: {len(confirmed)}")
print(f"Unique firms (by CIK): {confirmed['cik'].nunique()}")
confirmed[["accession_no", "cik", "ticker", "company_name", "file_date"]].head()

Confirmed events: 459
Unique firms (by CIK): 262


,accession_no,cik,ticker,company_name,file_date
49,0001652044-23-000013:googexhibit991q42022.htm,1652044,GOOG,"Alphabet Inc. (GOOG, GOOGL) (CIK 0001652044)",2023-02-02
289,0001326801-23-000063:meta03312023-exhibit991.htm,1326801,META,"Meta Platforms, Inc. (META) (CIK 0001326801)",2023-04-26
442,0000019617-23-000388:investordaypresentation.htm,19617,JPM,"JPMORGAN CHASE & CO (JPM, AMJB, JPM-PC, JPM-P...",2023-05-22
465,0001108524-23-000021:crm-q1fy24xexhibit991.htm,1108524,CRM,"Salesforce, Inc. (CRM) (CIK 0001108524)",2023-05-31
1450,0000072971-24-000004:ex993-wellsfargo4q23pres.htm,72971,WFC,"WELLS FARGO & COMPANY/MN (WFC, WFCNP, WFC-PA,...",2024-01-12


## Cell 5 — SEC XBRL fetch + point-in-time extraction functions

Fetches each unique firm's full XBRL Company Facts JSON **once** (cached in memory), then extracts whatever value was true as of each individual event's `file_date` locally - so a firm with 3 confirmed events costs 1 API call, not 3, while still giving each event its own correct point-in-time snapshot.

In [5]:
import requests
import time


def get_company_facts(cik, headers, timeout=20):
    """SEC's free structured financial-statement API, keyed by CIK."""
    cik10 = str(int(cik)).zfill(10)
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"
    try:
        resp = requests.get(url, headers=headers, timeout=timeout)
        if resp.status_code != 200:
            return None
        return resp.json()
    except Exception as e:
        print(f"  fetch error for CIK {cik}: {e}")
        return None


def extract_latest_value(facts, tag, as_of_date, unit="USD"):
    """Most recent reported value for `tag` with an end date on/before the event
    date, restricted to 10-K/10-Q filings - never leaks post-event information."""
    if facts is None:
        return None
    try:
        units = facts["facts"]["us-gaap"][tag]["units"][unit]
    except (KeyError, TypeError):
        return None

    as_of_str = as_of_date.strftime("%Y-%m-%d") if hasattr(as_of_date, "strftime") else str(as_of_date)
    candidates = [
        u for u in units
        if u.get("end", "9999-99-99") <= as_of_str and u.get("form") in VALID_FORMS
    ]
    if not candidates:
        return None
    return max(candidates, key=lambda u: u["end"])["val"]


REVENUE_TAGS = ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"]


def extract_revenue(facts, as_of_date):
    for tag in REVENUE_TAGS:
        val = extract_latest_value(facts, tag, as_of_date)
        if val is not None:
            return val
    return None


print("XBRL fetch/extraction functions loaded.")

XBRL fetch/extraction functions loaded.


## Cell 6 — Fetch once per firm, extract once per event

In [6]:
import numpy as np

unique_ciks = confirmed["cik"].dropna().unique()
print(f"Fetching SEC company facts for {len(unique_ciks)} unique firms ...")

facts_cache = {}
for i, cik in enumerate(unique_ciks, start=1):
    facts_cache[cik] = get_company_facts(cik, HEADERS)
    if i % 25 == 0:
        print(f"  fetched {i}/{len(unique_ciks)}")
    time.sleep(0.15)   # SEC fair-access guideline (~10 req/sec max)

n_failed = sum(1 for v in facts_cache.values() if v is None)
print(f"\nDone. {len(unique_ciks) - n_failed}/{len(unique_ciks)} firms returned data "
      f"({n_failed} failed - typically foreign private issuers who don't file XBRL "
      f"in us-gaap taxonomy, or very small/recently-registered filers).")

rows = []
for _, r in confirmed.iterrows():
    facts = facts_cache.get(r["cik"])
    assets = extract_latest_value(facts, "Assets", r["file_date"])
    liabilities = extract_latest_value(facts, "Liabilities", r["file_date"])
    revenue = extract_revenue(facts, r["file_date"])
    rd_expense = extract_latest_value(facts, "ResearchAndDevelopmentExpense", r["file_date"])

    rows.append({
        "accession_no": r["accession_no"],
        "cik": r["cik"],
        "ticker": r["ticker"],
        "event_date": r["file_date"],
        "total_assets": assets,
        "total_liabilities": liabilities,
        "revenue": revenue,
        "rd_expense": rd_expense,
        "log_assets": np.log(assets) if assets and assets > 0 else None,
        "leverage_ratio": (liabilities / assets) if assets and liabilities and assets > 0 else None,
        "rd_intensity": (rd_expense / revenue) if revenue and rd_expense and revenue > 0 else None,
    })

firm_financials = pd.DataFrame(rows)
firm_financials.head()

Fetching SEC company facts for 262 unique firms ...
  fetched 25/262
  fetched 50/262
  fetched 75/262
  fetched 100/262
  fetched 125/262
  fetched 150/262
  fetched 175/262
  fetched 200/262
  fetched 225/262
  fetched 250/262

Done. 262/262 firms returned data (0 failed - typically foreign private issuers who don't file XBRL in us-gaap taxonomy, or very small/recently-registered filers).


,accession_no,cik,ticker,event_date,total_assets,total_liabilities,revenue,rd_expense,log_assets,leverage_ratio,rd_intensity
0,0001652044-23-000013:googexhibit991q42022.htm,1652044,GOOG,2023-02-02,3.652640e+11,1.091200e+11,2.576370e+11,3.950000e+10,26.623886,0.298743,0.153316
1,0001326801-23-000063:meta03312023-exhibit991.htm,1326801,META,2023-04-26,1.844910e+11,5.969600e+10,3.892400e+10,9.381000e+09,25.940867,0.323571,0.241008
2,0000019617-23-000388:investordaypresentation.htm,19617,JPM,2023-05-22,3.744305e+12,3.441223e+12,1.286950e+11,NaN,28.951257,0.919055,NaN
3,0001108524-23-000021:crm-q1fy24xexhibit991.htm,1108524,CRM,2023-05-31,9.354100e+10,3.612900e+10,8.391984e+09,1.207000e+09,25.261666,0.386237,0.143828
4,0000072971-24-000004:ex993-wellsfargo4q23pres.htm,72971,WFC,2024-01-12,1.932468e+12,1.745025e+12,5.441500e+10,NaN,28.289819,0.903003,NaN


## Cell 7 — Prior AI-disclosure count (self-generated, no API call)

For each event, counts how many *earlier* confirmed AI events the same firm already had, using only data already in `screening_worksheet.csv` - satisfies the rubric's "public source or self-generated" constraint outright.

In [7]:
confirmed_sorted = confirmed.sort_values(["ticker", "file_date"]).copy()
confirmed_sorted["prior_ai_disclosure_count"] = confirmed_sorted.groupby("ticker").cumcount()

firm_financials = firm_financials.merge(
    confirmed_sorted[["accession_no", "prior_ai_disclosure_count"]],
    on="accession_no", how="left",
)

print(f"prior_ai_disclosure_count distribution:")
print(firm_financials["prior_ai_disclosure_count"].value_counts().sort_index())

prior_ai_disclosure_count distribution:
prior_ai_disclosure_count
0     262
1      81
2      36
3      19
4       9
5       7
6       6
7       3
8       2
9       1
10      1
11      1
12      1
13      1
14      1
15      1
16      1
17      1
18      1
19      1
20      1
21      1
22      1
23      1
24      1
25      1
26      1
27      1
28      1
29      1
30      1
31      1
32      1
33      1
34      1
35      1
36      1
37      1
38      1
39      1
40      1
41      1
42      1
Name: count, dtype: int64


## Cell 8 — Save results

In [8]:
firm_financials.to_csv(OUTPUT_FILE, index=False)

n = len(firm_financials)
for col in ["log_assets", "leverage_ratio", "rd_intensity"]:
    covered = firm_financials[col].notna().sum()
    print(f"{col} coverage: {covered}/{n} ({covered/n:.1%})")

print(f"\nSaved -> {OUTPUT_FILE}")

# --- Merge directly into analysis_dataset.csv (built by Step 5) so Step 6 picks
#     these up automatically, no separate manual merge step ---
if not os.path.exists(ANALYSIS_DATASET_FILE):
    print(f"\nWARNING: {ANALYSIS_DATASET_FILE} not found - run 05_market_model_car.ipynb "
          f"first, then re-run this cell to complete the merge.")
else:
    analysis_df = pd.read_csv(ANALYSIS_DATASET_FILE)

    new_cols = ["log_assets", "leverage_ratio", "rd_intensity", "prior_ai_disclosure_count"]
    # Drop these columns if a previous run of this notebook already added them,
    # so re-running doesn't create _x/_y duplicate columns
    analysis_df = analysis_df.drop(columns=[c for c in new_cols if c in analysis_df.columns])

    analysis_df = analysis_df.merge(
        firm_financials[["accession_no"] + new_cols],
        left_on="event_id", right_on="accession_no", how="left",
    ).drop(columns=["accession_no"])

    analysis_df.to_csv(ANALYSIS_DATASET_FILE, index=False)

    for col in new_cols:
        covered = analysis_df[col].notna().sum()
        print(f"analysis_dataset.csv: {col} coverage: {covered}/{len(analysis_df)} "
              f"({covered/len(analysis_df):.1%})")
    print(f"\nMerged into -> {ANALYSIS_DATASET_FILE}")

firm_financials.head()

log_assets coverage: 457/459 (99.6%)
leverage_ratio coverage: 442/459 (96.3%)
rd_intensity coverage: 294/459 (64.1%)

Saved -> /content/QM640-WALSH-CAPSTONE/data/raw/firm_financials.csv
analysis_dataset.csv: log_assets coverage: 443/445 (99.6%)
analysis_dataset.csv: leverage_ratio coverage: 429/445 (96.4%)
analysis_dataset.csv: rd_intensity coverage: 287/445 (64.5%)
analysis_dataset.csv: prior_ai_disclosure_count coverage: 445/445 (100.0%)

Merged into -> /content/QM640-WALSH-CAPSTONE/data/processed/analysis_dataset.csv


,accession_no,cik,ticker,event_date,total_assets,total_liabilities,revenue,rd_expense,log_assets,leverage_ratio,rd_intensity,prior_ai_disclosure_count
0,0001652044-23-000013:googexhibit991q42022.htm,1652044,GOOG,2023-02-02,3.652640e+11,1.091200e+11,2.576370e+11,3.950000e+10,26.623886,0.298743,0.153316,0
1,0001326801-23-000063:meta03312023-exhibit991.htm,1326801,META,2023-04-26,1.844910e+11,5.969600e+10,3.892400e+10,9.381000e+09,25.940867,0.323571,0.241008,0
2,0000019617-23-000388:investordaypresentation.htm,19617,JPM,2023-05-22,3.744305e+12,3.441223e+12,1.286950e+11,NaN,28.951257,0.919055,NaN,0
3,0001108524-23-000021:crm-q1fy24xexhibit991.htm,1108524,CRM,2023-05-31,9.354100e+10,3.612900e+10,8.391984e+09,1.207000e+09,25.261666,0.386237,0.143828,0
4,0000072971-24-000004:ex993-wellsfargo4q23pres.htm,72971,WFC,2024-01-12,1.932468e+12,1.745025e+12,5.441500e+10,NaN,28.289819,0.903003,NaN,0


## Commit and push results back to GitHub

In [9]:
!git -C {BASE_DIR} add "data/raw/firm_financials.csv"
!git -C {BASE_DIR} add "data/processed/analysis_dataset.csv"
!git -C {BASE_DIR} commit -m "Step 5b: firm-level financial variables via SEC XBRL, merged into analysis_dataset.csv"
!git -C {BASE_DIR} push

[main 56a46a2] Step 5b: firm-level financial variables via SEC XBRL, merged into analysis_dataset.csv
 2 files changed, 906 insertions(+), 446 deletions(-)
 create mode 100644 data/raw/firm_financials.csv
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (7/7), 46.05 KiB | 4.60 MiB/s, done.
Total 7 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   8c7ecb6..56a46a2  main -> main


## Done

`analysis_dataset.csv` now includes `log_assets`, `leverage_ratio`, `rd_intensity`, and `prior_ai_disclosure_count` alongside the existing `firm_size_log`/`sector` columns. Run `06_statistical_tests.ipynb` next - no further merging needed.